# Preparing a protein-ligand system for molecular dynamics simulation.

This is part 1 of a four-part tutorial on molecular dynamics simulations of biomolecular systems prepared for *CCPBioSim*'s Training week 2026.

* This part is concerned with preparing a molecular system for MD simulation.
* Part 2 deals with running an MD simulation in a terminal environment, using **AMBER**. 
* Part 3 shows how similar simulations can be run in a Python notebook environment, using the **OpenMM** package.
* Part 4 introduces methods for the analysis of data from MD simulations.

#### Authors:
* Charlie Laughton (charles.laughton@nottingham.ac.uk)
* (etc.)

### Orientation
Instructions for the workshop, and Python code for visualizing structures and other results, are in this window. The terminal window below is where you will run the commands that are refered to in the instructions (and shown as indented blocks).

### Prerequisites
If you are running this notebook on *CCPBioSim*'s training platform, all required software should be installed already. If you are running it locally, then if you have started this Notebook using the `run_notebook.sh` script in this folder, your Python environment should also be complete.

----------------

## Background
One of the most widespread uses of molecular dynamics simulations is to predict protein-ligand binding affinities, a key process in drug design and discovery. But before we can run any MD, we must design and build a three-dimensional model of the molecular system we wish to simulalate. There are many different ways to do this, but all involve more or less the same steps:

1. A molecular model for the heavy (non-hydrogen) atoms of the protein-ligand complex is generated from experimental data or predicted ab-initio.

2. The missing hydrogen atoms are added, taking into account issues like predicted tautomeric and ionization states, where relevant.

3. The all-atom model of the solute is immersed in a molecular model for a relevant solvent environment - typically water plus ions.

4. The complete molecular system is *parameterized*: from an analysis of the molecular structure, the force field parameters required to simulate its structure and dynamics are identified, and written out in a format suitable for the MD code that will be used to run the simulations.


In this tutorial you will explore deep-learning based structure-prediction tools for step 1, and then tools from the widely-used **AMBER** simulation package for steps 2-4.




## Step 1. Predicting the structure of a protein-ligand complex by deep learning.

Bcr-Abl is a oncogenic fusion protein with unregulated tyrosine kinase activity, resulting from a [genetic mutation](https://en.wikipedia.org/wiki/Philadelphia_chromosome) that is very frequently associated with chronic myologenous leukemia (CML). 

Your objective is to answer the question "Would you predict that the molecule shown below would inhibit Bcr-Abl, and so maybe be a useful anticancer drug?"

<img src="resources/imatinib_analogue_1.png" width="400">

The plan is to begin by predicting a likely structure of the complex between this molecule and the protein using a deep-learning approach (e.g. Alphafold3, Boltz2, etc.).

Although the details vary, in essence all these approaches require you to provide three pieces of information:

1. The sequence (1 letter amino acid code) for the protein.
2. A definition of the structure of the ligand (e.g. a SMILES string).
3. A multiple sequence alignment (MSA) for the protein (although usually there is an option to generate this on the fly instead).

### 1.1. The protein
So first we need the sequence of the protein. In a fresh browser tab navigate to the Uniprot Database [www.uniprot.org](http://www.uniprot.org) and type 'Bcr-Abl' into the search box.

 - The top search result should be entry *A9UF02*. Click on this.
 - When the page for this protein loads, click on *Family and Domains* in the left-hand column.
 - Change the value in the *Type* dropdown menu from 'All' to 'Domain'.
 - You should see three domains identified. The one you are interested in is the **protein kinase domain**.
 - Click on the *POSITION(S)* value for this ("756-1007"): the corresponding sequence will appear in a pop-up window.
 - This is the first piece of information you need.

Note there has been an experimental design decision here: instead of attempting to run a simulation of the whole of the Bcr-Abl protein, you are going to model only the kinase domain. This is a resource-saving simplification, but also probably not unreasonable in that a simple "wet lab" protein-ligand binding affinity experiment would almost certainly also be using an engineered version of just this domain, not full-length protein.

### 1.2 The ligand
We have been provided with an image of the chemical structure of the ligand, but we need to convert this into a more machine-readable form. There are various options here, but the most common will be to work out its [SMILES](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System) representation. We don't go into details of this here, but a valid SMILES for this ligand would be:
```
"c1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2ccncc2)n1"
```

However there is a wrinkle. Like the associated image, this SMILES representation does not explicity say where the hydrogen atoms are. While this is not a problem for many molecules most of the time because the positions of the hydrogen atoms are unambiguous, in this case that's not quite true. If the simulation is going to mimic a real "wet" experiment then it must model the ligand as it would be in a buffer solution at a pH of about 7.4, in which case one of the tertiary amine groups in the piperidine ring is going to be protonated.

The most likely protonated form for the ligand, and its corresponding SMILES representation, are shown below:

<img src="resources/imatinib_analogue_2.png" width="400">


### 1.3 The multiple sequence alignment (MSA)
The MSA provides information about patterns of evolutionary conservation and mutation in the sequence, which can be valuable extra information for structure prediction. In general you have two choices: 

1. provide the structure prediction tool with a "pre-built" MSA,
2. ask the program to construct one itself as a first step in the prediction workflow.

The latter option can be simpler, but MSA creation can be a time-consuming part of the process so if you have an existing one to hand, it can speed things up a lot.

For this reason we have supplied you with a pre-calculated MSA, it is in the (large!) file `msas/abl_paired_msa_chains_a.a3m` (take a look, if you like).


### 1.4 Structure prediction with Boltz2

We now have all the information required for a structure prediction. Here we will use [Boltz2](https://github.com/jwohlwend/boltz), but the procedure with Alphafold3, or some other tool, would be very similar.

The input file for Boltz2 is provided: `abl_ligand.yaml`. Take a look at it. You will see we have added the neccessary data discussed above: the sequence of the protein, the SMILES string for the ligand, and the path to the file with the pre-calculated MSA.

Now run the prediction, using the `mboltz` application provided. In the terminal window, make sure you are in the home folder, and then run the command:

```bash
./mboltz predict abl_ligand.yaml
```

The job should take about 30 seconds to complete. If all goes well, you will se a new folder created: `boltz_results_abl_ligand`.


*At this point an admission: the `mboltz` application provided here is a Boltz2 simulator - not the real thing! We have done this because installing and running these deep learning applications requires large GPUs and much additional software. The simulator provides the same 'look and feel' as the real Boltz2, but outputs what are actually pre-calculated results.*

### 1.5 Checking the Boltz2 prediction.

The structure of the results folder is as follows:
```
boltz_results_abl_ligand
├── lightning_logs                              # folder with log files - not important for now
├── msa                                         # empty folder in this case
├── predictions                                 # key outputs - one sub-folder per prediction
│   └── abl_ligand                              # we only asked for one prediction - here it is
│       ├── abl_ligand_model_0.cif              # the predicted structure, in cif format
│       ├── confidence_abl_ligand_model_0.json  # a summary of prediction confidence estimates
│       ├── pae_abl_ligand_model_0.npz          # detailed confidence metric - PAE scores (see later)
│       ├── pde_abl_ligand_model_0.npz          # detailed confidence metric - PDE scores (see later)
│       └── plddt_abl_ligand_model_0.npz        # detailed confidence metric - pLDDT scores (see later)
└── processed                                   # folder with processed versions of the job input 
```

#### 1.5.1 Visualization of the structure

We can view the prediucted structure in this notebok using nglview:

In [1]:
import nglview as nv
view = nv.show_file('boltz_results_abl_ligand/predictions/abl_ligand/abl_ligand_model_0.cif')
view

NGLWidget()

You should be able to see the molecule of the ligand nestled into a cleft between the N- and C-terminal lobes of the abl protein.

But how reliable is this prediction?


#### 1.5.2 Confidence metrics: pLDDT

The simplest confidence metric is the [predicted local distance difference test (pLDDt) score](https://www.ebi.ac.uk/training/online/courses/alphafold/inputs-and-outputs/evaluating-alphafolds-predicted-structures-using-confidence-scores/plddt-understanding-local-confidence/). Basically, this is a per-residue (or per-atom, for ligands) number between 0-100, and the higher it is the more confident the prediction. We can colour-code the structure by pLDDt value, using a scale which has become fairly standard:

| pLDDT range | interpretation | colour code |
|-------------|----------------|-------------|
| 90-100      | very high - both mainchain and side chain probably correct | blue |
| 70-90       | high - mainchain probably correct, side chain may not be | cyan |
| 50-70       | low - there may be erros in botht eh main chain and side chain | yellow |
| below 50    | very low - the structure here may be completely wrong | red |


To get nglviewer to adopt this colour-coding requires a bit of work, but here;s the code:

In [2]:
# A function to apply the "standard" pLDDt color scheme:
js_function = """
this.atomColor = function (atom) {
    if (atom.bfactor > 90) {
        return 0x0000FF // blue
    } else if (atom.bfactor > 70) {
        return 0x00FFFF // cyan
    } else if (atom.bfactor > 50) {
        return 0xFFFF00 // yellow  
    } else {
        return 0xFF0000 // red
    }
}
"""
nv.color.ColormakerRegistry.add_scheme_func("pLDDt", js_function)
#Load the model and color by pLDDt:
view = nv.show_file('boltz_results_abl_ligand/predictions/abl_ligand/abl_ligand_model_0.cif',
                    default_representation=False)
view.add_ribbon("protein", color_scheme="pLDDt")
view.add_ball_and_stick("ligand", color_scheme="pLDDt")
# view.layout.height = "500px" # Make the image a bit bigger than standard
view

NGLWidget()

You can see that Boltz2 is pretty confident about most of the prediction, both for the protein and the ligand, though there are a couple of lower-confidence regions - as these are in loops, quite possibly thay are due to intrinsic flexibility in thses regions.

#### 1.5.3 More confidence metrics: PAE and PDE

While *pLDDt* is a measure of the confidence in the preiction of an individual residue (or ligand atom), there are two other metrics that are concerned with the accuracy with which the spatial relationship between every residue/atom and every other residue/atom has been predicted - the *predicted alignment error (PAE)* and the *predicted distance error (PDE)*. If you want details, see [here](https://elanapearl.github.io/blog/2024/the-illustrated-alphafold/). 

Since these are pairwise metrics, they are best visualised in square matrix form. Here we plot the PDE results:

In [3]:
import numpy as np
from matplotlib import pyplot as plt

pde = np.load('boltz_results_abl_ligand/predictions/abl_ligand/pde_abl_ligand_model_0.npz')
plt.imshow(pde['pde'])
plt.colorbar()

FileNotFoundError: [Errno 2] No such file or directory: 'boltz_results_abl_ligand/predictions/abl_ligand/pde_abl_ligand_model_0.npz'

The smaller the PDE, the more confident the prediction of the relative position of the two residues/atoms. There is one row/column for each **residue** in the protein, plus one row/column for each **atom** in the ligand. In this particular case the PDE plot adds little to what was evident from the pLDDt data (that there is a flexible, difficult to predict region in the middle oif the protein, but the rest is confidently predicted), but in other cases these metrics can provide useful additional information.

#### 1.5.4 Validation against experimental data

A completely different approach to validation of a Boltz2 model is to ask "is there actually an experimentally-derived structure for this, or a closely related, protein-ligand complex that I can compare the prediction with?".

It's beyond the scope of this workshop to go into how you might actually do the search to check this, but a search of the [Protein Data Bank (PDB)](https://www.ebi.ac.uk/pdbe/) would lead you to a number of possible candidates, including entry [2HYY](https://www.ebi.ac.uk/pdbe/entry/pdb/2hyy).

Here we superimpose the Boltz2 predicted structure over that in the 2hyy crystal structure, which is of the right protein (abl) but with a different, but related, ligand (the anti cancer drug imatinib).

In [4]:
# Load the structures using MDTraj:
import mdtraj as mdt

prediction = mdt.load('boltz_results_abl_ligand/predictions/abl_ligand/abl_ligand_model_0.cif')
crystal = mdt.load('2hyy.pdb')
# The crystal structure contains 4 copies of the protein and ligand, we only want one of them:
crystal = crystal.atom_slice(crystal.topology.select('chainid 0 or chainid 4'))

# The protein model from the Boltz2 prediction includes a few extra residues at the N- and C-terminii
# of the protein compared to the crystal structure, so a bit of tweaking is needed for the superimposition:
pred_ca = prediction.topology.select('name CA')[2:-1]
crystal_ca = crystal.topology.select('name CA')
crystal_superposed = crystal.superpose(prediction, atom_indices = crystal_ca, ref_atom_indices = pred_ca)
# Now we can show the overlay - Boltz2 prediction in cyan, 2hyy structure in magenta:
view = nv.show_mdtraj(prediction)
xview = view.add_component(crystal_superposed)
view.clear_representations()
xview.clear_representations()
view.add_ribbon(color="cyan")
xview.add_ribbon(color="magenta")
view.add_ball_and_stick("ligand")
xview.add_ball_and_stick("ligand")
view.add_ball_and_stick("ligand and _C", color="cyan")
xview.add_ball_and_stick("ligand and _C", color="magenta")
# view.layout.height = "500px"
view

NGLWidget()

The overlay is pretty convincing, giving us extra confidence in the plausibility of this model for the protein-ligand complex.

We are now ready to move on to Step 2: adding missing hydrogen atoms to both the protein and ligand.

 ## Step 2. Protein–ligand system preparation for AMBER MD

This section converts the protein–ligand structure predicted by Boltz from
CIF format into a simulation-ready AMBER system.

The workflow includes:

1. Extracting the protein and ligand from the predicted CIF structure.
2. Protonating the protein.
3. Rebuilding the ligand chemical structure and adding hydrogens.
4. Assigning GAFF2 atom types and AM1-BCC charges to the ligand.
5. Combining the protein and ligand and solvating the complex using OPC water.
6. Neutralising the system and adding approximately 0.15 M NaCl.
7. Generating AMBER topology and coordinate files.
8. Running a short minimisation testand visualising the prepared and minimised structures.


Molecular simulation setup involves many individual decisions and transformations, and small errors introduced during preparation can affect everything that follows. In this step, we therefore use Python wherever practical to make the preparation workflow reproducible, transparent and easy to validate.

Each stage is designed to document what was done, make the workflow reproducible, check intermediate results before proceeding, fail clearly when something goes wrong, and retain intermediate files so that the preparation process can be traced.

As you work through the tutorial, take a moment to look at both the preparation steps and the checks that follow them. These checks help confirm that each stage has worked as expected before moving on, giving us confidence in the final simulation system.

### 2.1 Check the required software

 Before starting, we will confirm that the main dependencies are available.

The preparation uses:

* **MDTraj** to read, separate and validate molecular structures
* **RDKit** to reconstruct the ligand chemistry and add hydrogens
* **NGLView** to visualise structures in the notebook
* **PDB2PQR/PropKa** to assign protein protonation states
* **Reduce** to optimise protein hydrogen positions
* **pdb4amber** to prepare AMBER-compatible protein structures
* **Antechamber** and **parmchk2** to parameterise the ligand
* **LEaP** to combine, solvate and ionise the system
* **Sander** to perform a short minimisation test
* **ambpdb** to convert AMBER coordinates to PDB format



In [32]:
from pathlib import Path
from collections import Counter
import shutil
import subprocess
import sys

import numpy as np
import mdtraj as md
import nglview as nv
from rdkit import Chem, rdBase
from rdkit.Chem import AllChem


print("Python packages")
print(f"  Python:  {sys.version.split()[0]}")
print(f"  NumPy:   {np.__version__}")
print(f"  MDTraj:  {md.__version__}")
print(f"  NGLView: {nv.__version__}")
print(f"  RDKit:   {rdBase.rdkitVersion}")


required_commands = [
    "pdb2pqr",
    "reduce",
    "pdb4amber",
    "antechamber",
    "parmchk2",
    "tleap",
    "sander",
    "ambpdb",
]

print("\nCommand-line tools")

missing = []

for command in required_commands:
    location = shutil.which(command)
    print(f"  {command:12s} {location or 'NOT FOUND'}")

    if location is None:
        missing.append(command)

if missing:
    raise RuntimeError(
        "Missing required programs: " + ", ".join(missing)
    )

print("\nAll required dependencies were found.")

Python packages
  Python:  3.13.14
  NumPy:   2.5.3
  MDTraj:  1.11.1
  NGLView: 4.0.1
  RDKit:   2026.03.1

Command-line tools
  pdb2pqr      /opt/conda/bin/pdb2pqr
  reduce       /opt/conda/bin/reduce
  pdb4amber    /opt/conda/bin/pdb4amber
  antechamber  /opt/conda/bin/antechamber
  parmchk2     /opt/conda/bin/parmchk2
  tleap        /opt/conda/bin/tleap
  sander       /opt/conda/bin/sander
  ambpdb       /opt/conda/bin/ambpdb

All required dependencies were found.


### 2.2 Set up the preparation workflow

The main preparation choices are defined here so that they are used consistently throughout the workflow.

We will use:

* **pH 7.4 for protein protonation.** Protonation states determine the charges and hydrogen-bonding behaviour of ionisable residues. PDB2PQR and PropKa will estimate suitable states at a near-physiological pH.

* **AMBER ff19SB for the protein.** This force field provides the parameters used to describe the protein atoms, bonds, angles, torsions and non-bonded interactions.

* **GAFF2 with AM1-BCC partial charges for the ligand.** Protein force fields do not contain parameters for most drug-like molecules. GAFF2 supplies the ligand atom and bonded parameters, while AM1-BCC estimates its atomic partial charges.

* **OPC water.** OPC is a four-site explicit water model designed to reproduce important properties of liquid water and is commonly used with ff19SB.

* **A 12 Å truncated-octahedral solvent buffer.** The solvent box will extend at least 12 Å from the solute, reducing unwanted interactions between the protein–ligand complex and its periodic images. A truncated octahedron generally requires fewer water molecules than a rectangular box.

* **Approximately 0.15 M NaCl.** The system will first be neutralised and then additional sodium and chloride ions will be added to represent a physiologically relevant ionic strength.

Generated files will be organised into separate directories for protein preparation, ligand preparation, final system construction, validation and log files.



In [33]:
# Preparation settings
PH = 7.4
LIGAND_CHARGE = 1
WATER_BUFFER_ANGSTROM = 12.0
SALT_MOLARITY = 0.15

# Output directories
output_directories = [
    Path("02_protein_prep"),
    Path("03_ligand_prep"),
    Path("04_system"),
    Path("05_test"),
    Path("logs"),
]

for directory in output_directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Preparation settings")
print(f"  Protonation pH: {PH}")
print(f"  Ligand charge:  {LIGAND_CHARGE:+d}")
print(f"  Water model:    OPC")
print(f"  Solvent buffer: {WATER_BUFFER_ANGSTROM:.1f} Å")
print(f"  Salt target:    {SALT_MOLARITY:.2f} M")
print("\nOutput directories are ready.")

Preparation settings
  Protonation pH: 7.4
  Ligand charge:  +1
  Water model:    OPC
  Solvent buffer: 12.0 Å
  Salt target:    0.15 M

Output directories are ready.


### 2.3 Separate the protein and ligand

The Boltz prediction contains the protein and ligand in a single CIF file. These components must be separated because they require different preparation and parameterisation procedures.

MDTraj will be used to:

1. load the predicted complex;
2. select the protein atoms;
3. select the ligand using its residue name, `LIG1`; and
4. write each component to a separate PDB file.

Atom serial numbers and residue numbers are converted to integers before writing the PDB files. This avoids formatting errors caused by some identifiers imported from ModelCIF files.

The following files will be created:

* `02_protein_prep/protein_raw.pdb`
* `03_ligand_prep/ligand_pose.pdb`


In [34]:
# Load the Boltz-predicted protein–ligand complex.
cif_file = Path("boltz_results_abl_ligand/predictions/abl_ligand/abl_ligand_model_0.cif")


prediction = md.load(str(cif_file))

# Select the protein and ligand.
protein_atoms = prediction.topology.select("protein")
ligand_atoms = prediction.topology.select("resname LIG1")


protein = prediction.atom_slice(protein_atoms)
ligand = prediction.atom_slice(ligand_atoms)


def prepare_pdb_identifiers(trajectory):
    """Assign valid integer identifiers for PDB output."""

    for atom_number, atom in enumerate(
        trajectory.topology.atoms,
        start=1,
    ):
        atom.serial = atom_number

    for residue_number, residue in enumerate(
        trajectory.topology.residues,
        start=1,
    ):
        residue.resSeq = residue_number


prepare_pdb_identifiers(protein)
prepare_pdb_identifiers(ligand)

protein_file = Path("02_protein_prep/protein_raw.pdb")
ligand_file = Path("03_ligand_prep/ligand_pose.pdb")

protein.save_pdb(str(protein_file))
ligand.save_pdb(str(ligand_file))

print(f"Protein atoms: {protein.n_atoms}")
print(f"Ligand atoms:  {ligand.n_atoms}")
print()
print(f"Saved protein: {protein_file}")
print(f"Saved ligand:  {ligand_file}")

Protein atoms: 2169
Ligand atoms:  36

Saved protein: 02_protein_prep/protein_raw.pdb
Saved ligand:  03_ligand_prep/ligand_pose.pdb


### 2.4 Prepare the protein

The extracted protein must be standardised, protonated and checked before it can be used with the AMBER force field.

The preparation consists of four stages:

1. standardise the structure and add missing atoms within existing residues using `pdb4amber`;
2. assign protonation states at pH 7.4 using PDB2PQR and PropKa;
3. optimise hydrogen positions using Reduce; and
4. perform a final `pdb4amber` pass to produce an AMBER-compatible protein structure.

#### 2.4.1 Initial protein cleanup

`pdb4amber` standardises atom and residue names and adds missing atoms where possible using AMBER residue templates. It does not reconstruct missing residues or unresolved loops.

Connectivity records are omitted because LEaP will later construct the standard protein connectivity from its residue templates.


In [35]:
input_pdb = Path("02_protein_prep/protein_raw.pdb")
output_pdb = Path("02_protein_prep/protein_complete.pdb")
log_file = Path("logs/pdb4amber_initial.log")

command = [
    "pdb4amber",
    "-i", str(input_pdb),
    "-o", str(output_pdb),
    "--add-missing-atoms",
    "--no-conect",
]

with log_file.open("w") as log:
    result = subprocess.run(
        command,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

if result.returncode != 0:
    raise RuntimeError(
        "Initial pdb4amber preparation failed. "
        f"Check {log_file}"
    )

if not output_pdb.exists() or output_pdb.stat().st_size == 0:
    raise RuntimeError(
        f"pdb4amber did not create a valid output file: {output_pdb}"
    )

# Load the cleaned structure before accessing its properties.
complete_protein = md.load(str(output_pdb))

print("Initial protein cleanup completed.")
print(f"Atoms before cleanup: {protein.n_atoms}")
print(f"Atoms after cleanup:  {complete_protein.n_atoms}")
print(
    f"Protein residues:     "
    f"{complete_protein.topology.n_residues}"
)
print(f"Cleaned structure:    {output_pdb}")

Initial protein cleanup completed.
Atoms before cleanup: 2169
Atoms after cleanup:  4300
Protein residues:     266
Cleaned structure:    02_protein_prep/protein_complete.pdb


#### 2.4.2 Assign protonation states

Hydrogen atoms are usually absent from experimentally determined structures and may also be missing from predicted structures. They must be added before molecular dynamics because protonation states determine the formal charge of each residue and influence electrostatic interactions, hydrogen bonding, salt bridges and ligand binding.

In this workflow, **PDB2PQR** uses **PropKa** to estimate residue pKa values from the local structural environment. These estimates are then used to assign protonation states at **pH 7.4**. The AMBER naming convention is requested so that the resulting residue names can be recognised later by `pdb4amber` and `tleap`.

Important AMBER protonation-state names include:

| Residue name | Protonation state                      |
| ------------ | -------------------------------------- |
| `ASP`        | Deprotonated aspartate, charge −1      |
| `ASH`        | Protonated aspartate, neutral          |
| `GLU`        | Deprotonated glutamate, charge −1      |
| `GLH`        | Protonated glutamate, neutral          |
| `HID`        | Neutral histidine protonated at Nδ1    |
| `HIE`        | Neutral histidine protonated at Nε2    |
| `HIP`        | Doubly protonated histidine, charge +1 |
| `LYS`        | Protonated lysine, charge +1           |
| `LYN`        | Neutral lysine                         |
| `CYS`        | Neutral cysteine with a thiol hydrogen |
| `CYM`        | Deprotonated cysteine, charge −1       |

PDB2PQR produces two useful files:

* a `.pqr` file containing coordinates, atomic charges and radii;
* a `.pdb` file containing the protonated structure with AMBER-compatible residue and atom names.

The PDB file is used for the subsequent preparation steps. Ligand protonation and charge assignment are handled separately during ligand preparation.



In [36]:
from pathlib import Path
import subprocess

input_pdb = Path("02_protein_prep/protein_complete.pdb")
pdb2pqr_pdb = Path("02_protein_prep/protein_pdb2pqr.pdb")
pqr_file = Path("02_protein_prep/protein_pH7_4.pqr")
log_file = Path("logs/pdb2pqr.log")

if not input_pdb.exists():
    raise FileNotFoundError(
        f"Input structure not found: {input_pdb}"
    )

command = [
    "pdb2pqr",
    "--ff=AMBER",
    "--ffout=AMBER",
    "--titration-state-method=propka",
    f"--with-ph={PH}",
    "--keep-chain",
    f"--pdb-output={pdb2pqr_pdb}",
    str(input_pdb),
    str(pqr_file),
]

print(f"Running PDB2PQR protonation at pH {PH}...")

with log_file.open("w") as log:
    result = subprocess.run(
        command,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

if result.returncode != 0:
    raise RuntimeError(
        "PDB2PQR protonation failed. "
        f"Check {log_file}"
    )


print("PDB2PQR protonation completed.")
print(f"Protonated PDB: {pdb2pqr_pdb}")
print(f"PQR output:     {pqr_file}")
print(f"Log file:       {log_file}")

Running PDB2PQR protonation at pH 7.4...
PDB2PQR protonation completed.
Protonated PDB: 02_protein_prep/protein_pdb2pqr.pdb
PQR output:     02_protein_prep/protein_pH7_4.pqr
Log file:       logs/pdb2pqr.log


#### 2.4.3 Review the protonation assignments

Histidine can adopt several protonation states, represented by different AMBER residue names:

* `HID`: protonated on the delta nitrogen
* `HIE`: protonated on the epsilon nitrogen
* `HIP`: protonated on both nitrogens and positively charged

The following less-common assignments may also appear:

* `ASH`: protonated aspartic acid
* `GLH`: protonated glutamic acid
* `LYN`: neutral lysine
* `CYM`: negatively charged cysteine

These assignments should be reviewed carefully, particularly for residues in or near the ligand-binding site.


#### 2.4.4 Optimise hydrogen positions with Reduce

PDB2PQR determines which protonation states are appropriate at the selected pH and adds the corresponding hydrogen atoms. However, the precise positions and orientations of those hydrogens may still require optimisation.

**Reduce** evaluates the local hydrogen-bonding and steric environment to improve hydrogen placement. In this workflow, it is used to optimise hydrogen coordinates rather than to perform a second pH-based protonation-state prediction. The AMBER residue names assigned by PDB2PQR carry the selected protonation states into this step.

Hydrogen preparation is performed in two stages:

1. `reduce -Trim` removes the hydrogen atoms previously written by PDB2PQR.
2. `reduce -BUILD` rebuilds the hydrogens and optimises their positions using the local molecular environment.

Removing the existing hydrogens first avoids duplicate atoms and ensures that the final hydrogen coordinates are generated consistently by Reduce. The resulting structure should retain protonation-state names such as `HID`, `HIE`, `HIP`, `ASH` or `GLH`.

Following step confirms that the output contains valid coordinate records, verifies that trimming removed the hydrogens and checks that the build step added them again.

The resulting hydrogen-optimised protein will be standardised once more with `pdb4amber` before it is combined with the parameterised ligand.


In [37]:
from pathlib import Path
import subprocess

input_pdb = Path("02_protein_prep/protein_pdb2pqr.pdb")
trimmed_pdb = Path("02_protein_prep/protein_trimmed.pdb")
reduced_pdb = Path("02_protein_prep/protein_reduce.pdb")

if not input_pdb.exists():
    raise FileNotFoundError(input_pdb)


# Remove the hydrogens added by PDB2PQR.
trim_result = subprocess.run(
    ["reduce", "-Trim", str(input_pdb)],
    capture_output=True,
    text=True,
)

trimmed_pdb.write_text(trim_result.stdout)
Path("logs/reduce_trim.log").write_text(trim_result.stderr)

trimmed_atoms = sum(
    line.startswith(("ATOM  ", "HETATM"))
    for line in trim_result.stdout.splitlines()
)

if trimmed_atoms == 0:
    raise RuntimeError(
        "Reduce trimming failed. "
        "Check logs/reduce_trim.log."
    )

print(f"Trimmed structure: {trimmed_pdb}")
print(f"Atoms after trimming: {trimmed_atoms}")


# Rebuild and optimise hydrogen positions.
build_result = subprocess.run(
    ["reduce", "-BUILD", str(trimmed_pdb)],
    capture_output=True,
    text=True,
)

reduced_pdb.write_text(build_result.stdout)
Path("logs/reduce_build.log").write_text(build_result.stderr)

rebuilt_atoms = sum(
    line.startswith(("ATOM  ", "HETATM"))
    for line in build_result.stdout.splitlines()
)

if rebuilt_atoms == 0:
    raise RuntimeError(
        "Reduce hydrogen building failed. "
        "Check logs/reduce_build.log."
    )


# Count the rebuilt hydrogens.
rebuilt_hydrogens = sum(
    line[76:78].strip() == "H"
    or line[12:16].strip().startswith("H")
    for line in reduced_pdb.read_text().splitlines()
    if line.startswith(("ATOM  ", "HETATM"))
)

if rebuilt_hydrogens == 0:
    raise RuntimeError(
        "Reduce did not rebuild any hydrogens."
    )

print(f"Reduced structure: {reduced_pdb}")
print(f"Atoms after rebuilding: {rebuilt_atoms}")
print(f"Hydrogens rebuilt: {rebuilt_hydrogens}")

Trimmed structure: 02_protein_prep/protein_trimmed.pdb
Atoms after trimming: 2170
Reduced structure: 02_protein_prep/protein_reduce.pdb
Atoms after rebuilding: 4258
Hydrogens rebuilt: 2088


#### 2.4.5 Standardise the final protein structure

The hydrogen-optimised structure produced by Reduce is passed through `pdb4amber` once more to ensure that its residue names, atom names and coordinate records are compatible with AMBER.

This final standardisation step does not perform another pH calculation. The protonation states assigned by PDB2PQR and the hydrogen positions generated by Reduce should be retained.

The `--no-conect` option prevents PDB `CONECT` records from being written. Protein connectivity will be generated later by `tleap` from the selected AMBER force field.

The resulting `protein_prepared.pdb` file will be used when constructing the protein–ligand system in `tleap`.


In [38]:
command = [
    "pdb4amber",
    "-i", "02_protein_prep/protein_reduce.pdb",
    "-o", "02_protein_prep/protein_prepared.pdb",
    "--no-conect",
]

with open("logs/pdb4amber_final.log", "w") as log:
    result = subprocess.run(
        command,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

if result.returncode != 0:
    raise RuntimeError(
        "Final pdb4amber preparation failed. "
        "Check logs/pdb4amber_final.log"
    )

print("Final prepared protein written to:")
print("02_protein_prep/protein_prepared.pdb")

Final prepared protein written to:
02_protein_prep/protein_prepared.pdb


### 2.5 Prepare the ligand

The ligand requires a separate preparation workflow because standard protein force fields do not contain parameters for arbitrary small molecules.

The ligand extracted from the predicted CIF file retains its binding pose, but the intermediate PDB format does not reliably describe bond orders, aromaticity, formal charges or ligand protonation. These chemical properties must be restored before AMBER atom types and partial charges can be assigned.

The ligand will be prepared by:

1. reconstructing its chemistry from a known SMILES string;
2. transferring the correct bond orders and formal charges to the predicted pose;
3. adding and optimising hydrogen atoms;
4. assigning GAFF2 atom types and AM1-BCC partial charges with Antechamber;
5. generating any missing bonded parameters with `parmchk2`;
6. checking that the calculated partial charges sum to the expected formal charge.


#### 2.5.1 Reconstruct the ligand chemistry and add hydrogens

The ligand PDB contains the predicted three-dimensional heavy-atom coordinates, but PDB files do not explicitly encode bond orders. Inferring the chemical structure from interatomic distances alone can therefore produce incorrect aromatic, single or double bonds.

A known SMILES string is used as a chemical template. RDKit transfers the bond orders and formal charges from this template to the ligand pose while retaining the predicted heavy-atom coordinates.

The `[NH+]` notation in the SMILES specifies the protonated nitrogen in the piperazine ring and gives this ligand a net formal charge of **+1** at the selected protonation state.

Explicit hydrogen atoms are then added. Only their positions will be optimised in the next step, so the predicted heavy-atom binding pose remains unchanged.



In [39]:
from rdkit import Chem
from rdkit.Chem import AllChem

ligand_smiles = ( "c1ccc(NC(=O)c2ccc(C[NH+]3CCN(C)CC3)cc2)cc1Nc1nccc(-c2ccncc2)n1")

template = Chem.MolFromSmiles(ligand_smiles)

if template is None:
    raise ValueError("RDKit could not interpret the ligand SMILES.")

pose = Chem.MolFromPDBFile(
    "03_ligand_prep/ligand_pose.pdb",
    removeHs=False,
    sanitize=False,
    proximityBonding=True,
)

if pose is None:
    raise ValueError("RDKit could not read the predicted ligand pose.")

pose.UpdatePropertyCache(strict=False)

try:
    ligand_chemistry = AllChem.AssignBondOrdersFromTemplate(template, pose)
except Exception as exc:
    raise RuntimeError(
        "The ligand SMILES could not be matched to the predicted pose. "
        "Check that the SMILES and the CIF ligand describe the same molecule."
    ) from exc

ligand_with_h = Chem.AddHs(
    ligand_chemistry,
    addCoords=True,
)

formal_charge = Chem.GetFormalCharge(ligand_with_h)

print(f"Template heavy atoms: {template.GetNumHeavyAtoms()}")
print(f"Pose heavy atoms:     {pose.GetNumHeavyAtoms()}")
print(f"Atoms with H:         {ligand_with_h.GetNumAtoms()}")
print(f"Formal charge:        {formal_charge:+d}")

Template heavy atoms: 36
Pose heavy atoms:     36
Atoms with H:         66
Formal charge:        +1


[14:49:34] WARNING: More than one matching pattern found - picking one



### 2.5.2 Optimise the added ligand hydrogens

The hydrogen coordinates generated by RDKit are initial estimates and may contain locally unfavourable bond lengths, angles or contacts. A short molecular-mechanics optimisation is therefore performed with the Universal Force Field (UFF).

Only hydrogen atoms are allowed to move during this optimisation. Every ligand heavy atom is fixed so that the binding pose predicted by Boltz is preserved.

The optimisation return code is interpreted as follows:

0: the optimisation converged\
1: the iteration limit was reached before full convergence

The prepared ligand is saved in two formats:

ligand_pose_H.mol retains bond orders, formal charges and hydrogen atoms and will be used as the input to Antechamber;
ligand_pose_H.pdb is provided for visualization and structural inspection

In [40]:
force_field = AllChem.UFFGetMoleculeForceField(ligand_with_h)

for atom in ligand_with_h.GetAtoms():
    if atom.GetAtomicNum() != 1:
        force_field.AddFixedPoint(atom.GetIdx())

force_field.Initialize()
status = force_field.Minimize(maxIts=500)

print(f"UFF hydrogen optimisation status: {status}")

mol_file = Path("03_ligand_prep/ligand_pose_H.mol")
pdb_file = Path("03_ligand_prep/ligand_pose_H.pdb")

Chem.MolToMolFile(ligand_with_h, str(mol_file))
Chem.MolToPDBFile(ligand_with_h, str(pdb_file))

print(f"Saved: {mol_file}")
print(f"Saved: {pdb_file}")

UFF hydrogen optimisation status: 0
Saved: 03_ligand_prep/ligand_pose_H.mol
Saved: 03_ligand_prep/ligand_pose_H.pdb


#### 2.5.3 Assign GAFF2 atom types and AM1-BCC charges

The prepared ligand is passed to **Antechamber** to generate an AMBER-compatible MOL2 file.

Antechamber performs two important tasks:

* assigns **GAFF2 atom types**, which connect the ligand atoms to the General AMBER Force Field;
* calculates **AM1-BCC partial atomic charges**, which describe the ligand’s electrostatic interactions.

The total formal charge must be supplied explicitly. For this ligand, `-nc 1` is used because the protonated piperazine nitrogen gives the molecule a net charge of **+1**.

The option `maxcyc=0` prevents SQM from performing an additional geometry optimisation during the AM1 calculation. This preserves the ligand pose that was already prepared with RDKit. It does not disable the electronic calculation required to obtain the AM1-BCC charges.

The resulting `LIG_gaff2.mol2` file contains:

* ligand coordinates;
* atom and bond information;
* GAFF2 atom types;
* AM1-BCC partial charges;
* the AMBER residue name `LIG`.


In [41]:
command = [
    "antechamber",
    "-i", "03_ligand_prep/ligand_pose_H.mol",
    "-fi", "mdl",
    "-o", "03_ligand_prep/LIG_gaff2.mol2",
    "-fo", "mol2",
    "-at", "gaff2",
    "-c", "bcc",
    "-nc", "1",
    "-rn", "LIG",
    "-pf", "y",
    "-ek",
    'qm_theory="AM1", grms_tol=0.0005, '
    'scfconv=1.d-8, maxcyc=0',
]

with open("logs/antechamber.log", "w") as log:
    result = subprocess.run(
        command,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

if result.returncode != 0:
    raise RuntimeError(
        "Antechamber failed. Inspect logs/antechamber.log and sqm.out."
    )

ligand_mol2 = Path("03_ligand_prep/LIG_gaff2.mol2")

if not ligand_mol2.exists() or ligand_mol2.stat().st_size == 0:
    raise RuntimeError("Antechamber did not create LIG_gaff2.mol2.")

print(f"Created: {ligand_mol2}")

Created: 03_ligand_prep/LIG_gaff2.mol2


#### 2.5.4 Generate supplementary ligand parameters

GAFF2 contains parameters for a wide range of organic molecules, but the particular combination of atom types present in a ligand may require additional bonded parameters.

`parmchk2` examines the GAFF2 atom types assigned in the MOL2 file and checks whether suitable parameters are available for every bond, angle, dihedral and improper torsion. Any additional parameters required for this ligand are written to an AMBER force-field modification file, or `frcmod` file.

The option `-s 2` instructs `parmchk2` to use the **GAFF2** parameter database. This must be consistent with the `gaff2` atom types assigned by Antechamber.

The generated `LIG_gaff2.frcmod` file supplements GAFF2; it does not replace the main GAFF2 force field. Both will be loaded later in `tleap`.

Lines containing `ATTN` indicate parameters that were assigned using analogy and deserve additional review, particularly for unusual ligand chemistry.


In [42]:
command = [
    "parmchk2",
    "-i", "03_ligand_prep/LIG_gaff2.mol2",
    "-f", "mol2",
    "-o", "03_ligand_prep/LIG_gaff2.frcmod",
    "-s", "2",
]

with open("logs/parmchk2.log", "w") as log:
    result = subprocess.run(
        command,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

if result.returncode != 0:
    raise RuntimeError(
        "parmchk2 failed. Check logs/parmchk2.log"
    )

print("Created:")
print("03_ligand_prep/LIG_gaff2.frcmod")

Created:
03_ligand_prep/LIG_gaff2.frcmod


#### 2.5.5 Validate the ligand parameters

Before constructing the complete system, check that Antechamber and parmchk2 generated the expected ligand parameter files:

LIG_gaff2.mol2, containing the ligand coordinates, GAFF2 atom types and AM1-BCC partial charges;
LIG_gaff2.frcmod, containing any supplementary bonded parameters required by the ligand.

The partial charges in the MOL2 file should sum to the expected formal ligand charge of +1, allowing for small numerical rounding differences.

The frcmod file should also be inspected for lines containing ATTN. These indicate parameters assigned by analogy that may require further assessment, particularly for unusual functional groups or chemical environments.

The ligand geometry, bond orders, protonation state and formal charge should be chemically reasonable before proceeding. The ligand files will undergo an additional compatibility check when they are loaded with the protein force field in tleap.

### 2.6 Build the solvated protein–ligand system

The prepared protein and parameterised ligand can now be combined into a single AMBER system using `tleap`.

The following force fields and solvent model will be loaded:

* **ff19SB** for the protein;
* **GAFF2** and the ligand-specific `frcmod` file for the ligand;
* **OPC** for the water model and compatible ion parameters.

The complex will be placed in a truncated-octahedral water box with a **12 Å** buffer. A truncated octahedron generally requires fewer water molecules than a rectangular box for a roughly globular protein.


#### 2.6.1 Construct a preliminary neutralised system

A preliminary `tleap` build is used to determine how many water molecules are present after solvation. This is needed to estimate the number of NaCl pairs corresponding to approximately **0.15 M** salt.

The preliminary build performs the following operations:

1. loads the protein, water and ligand force fields;
2. loads the supplementary ligand parameters;
3. combines the prepared protein and parameterised ligand;
4. checks the unsolvated complex;
5. adds an OPC truncated-octahedral solvent box;
6. adds enough sodium ions to neutralise the system;
7. writes preliminary topology, coordinate and PDB files.

The command

```text
addIonsRand system Na+ 0
```

does not request zero sodium ions. In `tleap`, the zero requests that enough `Na+` ions be added to neutralise a negatively charged system. These counterions are separate from the additional NaCl pairs that will be added later to represent the salt concentration.


In [43]:
preliminary_leap = """\
source leaprc.protein.ff19SB
source leaprc.water.opc
source leaprc.gaff2

loadAmberParams 03_ligand_prep/LIG_gaff2.frcmod

protein = loadPdb 02_protein_prep/protein_prepared.pdb
ligand = loadMol2 03_ligand_prep/LIG_gaff2.mol2

system = combine {protein ligand}

check system
charge system

solvateOct system OPCBOX 12.0

charge system
addIonsRand system Na+ 0
charge system

check system

savePdb system 04_system/preliminary_neutralised.pdb
saveAmberParm system 04_system/preliminary.prmtop 04_system/preliminary.inpcrd

quit
"""

Path("04_system/tleap_preliminary.in").write_text(preliminary_leap)

print(Path("04_system/tleap_preliminary.in").read_text())

source leaprc.protein.ff19SB
source leaprc.water.opc
source leaprc.gaff2

loadAmberParams 03_ligand_prep/LIG_gaff2.frcmod

protein = loadPdb 02_protein_prep/protein_prepared.pdb
ligand = loadMol2 03_ligand_prep/LIG_gaff2.mol2

system = combine {protein ligand}

check system
charge system

solvateOct system OPCBOX 12.0

charge system
addIonsRand system Na+ 0
charge system

check system

savePdb system 04_system/preliminary_neutralised.pdb
saveAmberParm system 04_system/preliminary.prmtop 04_system/preliminary.inpcrd

quit



#### Run the preliminary tleap build

The generated LEaP input file is now executed with tleap. During this step, AMBER loads the selected force fields, combines the prepared protein and ligand, solvates the complex with OPC water and adds the counterions required to neutralise the system.

LEaP also checks whether atom types and force-field parameters are available for every part of the system. Its output is written to a log file, which should be inspected for messages such as Fatal Error, missing atom types or missing parameters.

The preliminary build produces:

- an AMBER topology file (.prmtop)
- an AMBER coordinate file (.inpcrd)
- a solvated PDB file for inspection and water counting.

Warnings should be reviewed, but they do not necessarily indicate that the build failed. A successful run should finish without fatal errors and create all three output files.

In [44]:
from pathlib import Path
import subprocess

leap_input = Path("04_system/tleap_preliminary.in")
leap_log = Path("logs/tleap_preliminary.log")

if not leap_input.exists():
    raise FileNotFoundError(f"Missing LEaP input: {leap_input}")

print(
    "Running the preliminary tleap build. Please wait...",
    flush=True,
)

with leap_log.open("w") as log:
    result = subprocess.run(
        [
            "tleap",
            "-f",
            str(leap_input),
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )


print(f"LEaP return code: {result.returncode}")
print(f"LEaP log: {leap_log}")

log_text = leap_log.read_text(errors="replace")

if result.returncode != 0 or "Fatal Error" in log_text:
    print("\n".join(log_text.splitlines()[-50:]))
    raise RuntimeError(
        "Preliminary LEaP build failed. "
        "Inspect logs/tleap_preliminary.log"
    )

print("Preliminary LEaP build completed.")

log_text = Path("logs/tleap_preliminary.log").read_text(
    errors="replace"
)

for line in log_text.splitlines():
    if any(
        phrase in line.lower()
        for phrase in [
            "total unperturbed charge",
            "unit is ok",
            "added",
            "fatal",
            "error",
            "exiting leap",
        ]
    ):
        print(line)

Running the preliminary tleap build. Please wait...
LEaP return code: 0
LEaP log: logs/tleap_preliminary.log
Preliminary LEaP build completed.
  Leap added 42 missing atoms according to residue templates:
Unit is OK.
Total unperturbed charge:  -7.003000
  Added 14808 residues.
Total unperturbed charge:  -7.003000
Total unperturbed charge:  -0.003000
Unit is OK.
Exiting LEaP: Errors = 0; Warnings = 203; Notes = 1.


#### 2.6.3 Calculate the number of NaCl pairs

For an approximate **0.15 M NaCl** concentration, the required number of ion pairs is estimated from the number of water molecules in the preliminary solvated system:

$$
N_{\mathrm{NaCl}}
=
\operatorname{round}
\left(
\frac{C_{\mathrm{NaCl}} \times N_{\mathrm{water}}}{55.56}
\right)
$$

where:

- $N_{\mathrm{NaCl}}$ is the number of NaCl pairs to add;
- $C_{\mathrm{NaCl}}$ is the target salt concentration in molar units;
- $N_{\mathrm{water}}$ is the number of water molecules;
- **55.56 M** is the approximate molar concentration of pure water.

Neutralising counterions and added NaCl pairs serve different purposes:

* The first ion command adds enough sodium ions to neutralise the negative charge of the protein–ligand system.
* The second ion command adds equal numbers of sodium and chloride ions to represent the bulk salt concentration.

The resulting concentration is approximate because the calculation is based on the number of water molecules rather than the exact equilibrated volume of the simulation box.


In [45]:
preliminary_pdb = Path(
    "04_system/preliminary_neutralised.pdb"
)

number_of_waters = sum(
    line.startswith(("ATOM  ", "HETATM"))
    and line[17:20].strip() in {"WAT", "HOH", "OPC"}
    and line[12:16].strip() in {"O", "OW"}
    for line in preliminary_pdb.read_text().splitlines()
)

salt_pairs = round(
    SALT_MOLARITY * number_of_waters / 55.56
)

print(f"Water molecules: {number_of_waters}")
print(f"Target NaCl:     {SALT_MOLARITY:.2f} M")
print(f"NaCl pairs:      {salt_pairs}")

Water molecules: 14801
Target NaCl:     0.15 M
NaCl pairs:      40


#### 2.6.4 Build the final solvated system with `tleap`

The final `tleap` build recreates the protein–ligand system using the selected force fields, OPC water and the calculated number of NaCl pairs.

Ions are added in two stages:

1. enough sodium counterions are added to neutralise the initial negative charge of the protein–ligand system;
2. equal numbers of sodium and chloride ions are added to represent approximately **0.15 M NaCl**.

The neutralising sodium ions and the added NaCl pairs serve different purposes. Adding equal numbers of sodium and chloride ions does not change the total charge, so the final system should remain electrically neutral.

LEaP then checks the complete system and writes:

* `abl_ligand.prmtop`, containing the topology and force-field parameters;
* `abl_ligand.inpcrd`, containing the coordinates and periodic box dimensions;
* `abl_ligand_solvated.pdb`, for visual inspection.

The LEaP log should be checked for fatal errors, unknown atom types and missing parameters before proceeding to the validation and minimisation test.


In [46]:
final_leap = f"""\
source leaprc.protein.ff19SB
source leaprc.water.opc
source leaprc.gaff2

loadAmberParams 03_ligand_prep/LIG_gaff2.frcmod

protein = loadPdb 02_protein_prep/protein_prepared.pdb
ligand = loadMol2 03_ligand_prep/LIG_gaff2.mol2

system = combine {{protein ligand}}

check system
charge system

solvateOct system OPCBOX 12.0

# Neutralise the system automatically.
addIonsRand system Na+ 0

# Add approximately 0.15 M NaCl.
addIonsRand system Na+ {salt_pairs} Cl- {salt_pairs}

charge system
check system

saveAmberParm system 04_system/abl_ligand.prmtop 04_system/abl_ligand.inpcrd
savePdb system 04_system/abl_ligand_solvated.pdb

quit
"""

Path("04_system/tleap_final.in").write_text(final_leap)

print(Path("04_system/tleap_final.in").read_text())

source leaprc.protein.ff19SB
source leaprc.water.opc
source leaprc.gaff2

loadAmberParams 03_ligand_prep/LIG_gaff2.frcmod

protein = loadPdb 02_protein_prep/protein_prepared.pdb
ligand = loadMol2 03_ligand_prep/LIG_gaff2.mol2

system = combine {protein ligand}

check system
charge system

solvateOct system OPCBOX 12.0

# Neutralise the system automatically.
addIonsRand system Na+ 0

# Add approximately 0.15 M NaCl.
addIonsRand system Na+ 40 Cl- 40

charge system
check system

saveAmberParm system 04_system/abl_ligand.prmtop 04_system/abl_ligand.inpcrd
savePdb system 04_system/abl_ligand_solvated.pdb

quit



#### Run the final tleap build

The final LEaP input file is now executed with tleap. This creates the fully solvated and neutralised protein–ligand system with the calculated number of NaCl pairs.
The output is written to logs/tleap_final.log. This log should be checked for fatal errors, unknown atom types and missing force-field parameters.
Warnings should be reviewed, but the presence of warnings alone does not necessarily mean the build failed. The topology and coordinate files must both exist and be non-empty before proceeding to system validation.

In [47]:
final_input = Path("04_system/tleap_final.in")
final_log = Path("logs/tleap_final.log")

print(
    "Running the final tleap build. Please wait...",
    flush=True,
)

with final_log.open("w") as log:
    result = subprocess.run(
        ["tleap", "-f", str(final_input)],
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

log_text = final_log.read_text(errors="replace")

if result.returncode != 0 or "Fatal Error" in log_text:
    print("\n".join(log_text.splitlines()[-50:]))
    raise RuntimeError(
        "Final LEaP build failed. Inspect logs/tleap_final.log"
    )

print("Final LEaP build completed successfully.")

Running the final tleap build. Please wait...
Final LEaP build completed successfully.


#### Review the final LEaP output

Before continuing, inspect the output from the final `tleap` run and the log file at `logs/tleap_final.log`.

Check that:

* no `Fatal Error` message is present;
* no atoms are reported as missing an atom type;
* no bond, angle or torsion parameters are reported as missing;
* `check system` finishes with `Unit is OK`;
* the final system charge is close to zero;
* the final line reports `Exiting LEaP: Errors = 0`;
* the topology, coordinate and PDB files have been created.

Warnings may still be reported. These should be reviewed, but warnings alone do not necessarily indicate that the system build failed.


### 2.7 Review the solvated system


The following cell loads the AMBER topology and coordinates, checks the system dimensions and ion composition, and displays the protein–ligand complex with the ions. Water is omitted from the visualisation for clarity.

In [52]:
from collections import Counter
import mdtraj as md
import nglview as nv

# Expected composition
target_salt_molarity = 0.15


# Load the AMBER system
system_trajectory = md.load(
    "04_system/abl_ligand.inpcrd",
    top="04_system/abl_ligand.prmtop",
)

# Count residues from the topology
counts = Counter(
    residue.name
    for residue in system_trajectory.topology.residues
)

sodium_count = sum(
    counts[name] for name in {"Na+", "NA", "Na"}
)
chloride_count = sum(
    counts[name] for name in {"Cl-", "CL", "Cl"}
)
water_count = sum(
    counts[name] for name in {"WAT", "HOH", "OPC"}
)

# Estimate the expected number of salt pairs
expected_salt_pairs = round(
    target_salt_molarity * water_count / 55.56
)



observed_salt_molarity = (
    chloride_count * 55.56 / water_count
)

print("System summary")
print(f"  Frames:       {system_trajectory.n_frames}")
print(f"  Atoms:        {system_trajectory.n_atoms}")
print(f"  Water:        {water_count}")
print(f"  Sodium:       {sodium_count} ")
print(f"  Chloride:     {chloride_count} ")
print(f"  Approx. NaCl: {observed_salt_molarity:.3f} M")

if system_trajectory.n_frames != 1:
    raise RuntimeError("Unexpected number of coordinate frames.")

# Display protein, ligand and ions; hide water for clarity
view = nv.show_mdtraj(system_trajectory)
view.clear_representations()
view.add_cartoon(selection="protein", color="cyan")
view.add_ball_and_stick(selection="[LIG]", color="orange")
view.add_spacefill(selection="_Na or _Cl", radius=0.5)
view.center(selection="protein or [LIG]")
view

System summary
  Frames:       1
  Atoms:        63337
  Water:        14721
  Sodium:       47 
  Chloride:     40 
  Approx. NaCl: 0.151 M


NGLWidget()

### 2.8 AMBER minimisation smoke test

A short energy minimisation is used as a smoke test for the generated topology
and coordinates. It checks that AMBER can read the system and evaluate its
energy without encountering missing parameters, numerical instability or
invalid coordinates.

This is not the complete equilibration protocol. A production workflow should
normally include a longer staged minimisation, heating, density equilibration
and pressure equilibration.

The formatted restart options `ioutfm=0` and `ntxo=1` are used so that the
result can be read directly by `ambpdb` and other text-restart readers.

In [53]:
minimisation_input = """\
Short minimisation test
&cntrl
  imin   = 1,
  maxcyc = 500,
  ncyc   = 250,
  ntb    = 1,
  cut    = 9.0,
  ntpr   = 50,
  ntwx   = 0,
  ioutfm = 0,
  ntxo   = 1,
/
"""
Path("05_test/min_check.in").write_text(minimisation_input)
print(Path("05_test/min_check.in").read_text())

Short minimisation test
&cntrl
  imin   = 1,
  maxcyc = 500,
  ncyc   = 250,
  ntb    = 1,
  cut    = 9.0,
  ntpr   = 50,
  ntwx   = 0,
  ioutfm = 0,
  ntxo   = 1,
/



#### 2.8.1 Run the minimisation with sander

The prepared AMBER system is now subjected to a short energy minimisation using sander. This serves as a smoke test to confirm that the topology, coordinates and force-field parameters can be read correctly and that the system energy can be evaluated without numerical or structural errors.

The run uses:

- min_check.in as the minimisation input;
- abl_ligand.prmtop as the system topology;
- abl_ligand.inpcrd as the starting coordinates.

The main outputs are:

- min_check.out, containing the energy and completion information;
- min_check.rst7, containing the minimised coordinates;
- min_check.mdinfo, containing a concise run summary.

A successful run should return an exit code of zero and create a non-empty restart file. The output should subsequently be checked for FINAL RESULTS, NaN, Infinity, FATAL or SANDER BOMB messages. This short minimisation validates the prepared system but does not replace a complete staged minimisation and equilibration protocol.

In [ ]:
command = [
    "sander",
    "-O",
    "-i", "05_test/min_check.in",
    "-o", "05_test/min_check.out",
    "-p", "04_system/abl_ligand.prmtop",
    "-c", "04_system/abl_ligand.inpcrd",
    "-r", "05_test/min_check.rst7",
    "-inf", "05_test/min_check.mdinfo",
]
print("Running short energy minimisation... please wait.")

result = subprocess.run(command)

if result.returncode != 0:
    raise RuntimeError(
        "The minimisation failed. Check 05_test/min_check.out"
    )

restart = Path("05_test/min_check.rst7")

if not restart.exists() or restart.stat().st_size == 0:
    raise RuntimeError("The minimised restart file was not created.")

print("Short minimisation completed.")
print(f"Restart file: {restart}")

Running short energy minimisation... please wait.


#### 2.8.2 Review and visualise the minimised system

The `sander` output is checked for normal completion and for signs of numerical or fatal errors. The final energy summary is then displayed.

If the minimisation completed successfully, `ambpdb` converts the formatted restart file into a PDB structure. The minimised protein–ligand complex is subsequently displayed with the protein in cyan, the ligand in orange and the ions shown as spheres. Water molecules remain in the system but are hidden for clarity.

In [31]:
from pathlib import Path
import subprocess
import nglview as nv

output_file = Path("05_test/min_check.out")
restart_file = Path("05_test/min_check.rst7")
topology_file = Path("04_system/abl_ligand.prmtop")
minimised_pdb = Path("05_test/min_check.pdb")
ambpdb_log = Path("logs/ambpdb.log")

# Check the sander output
min_output = output_file.read_text(errors="replace")

completion_terms = [
    "FINAL RESULTS",
    "Job began",
    "Run   done",
    "TIMINGS",
]

fatal_terms = [
    "NaN",
    "Infinity",
    "FATAL",
    "SANDER BOMB",
    "Segmentation fault",
]

print("Completion checks")
for term in completion_terms:
    print(f"  {term:15s}: {term in min_output}")

detected = [
    term for term in fatal_terms
    if term.lower() in min_output.lower()
]

if detected:
    raise RuntimeError(
        "Potential minimisation failure: " + ", ".join(detected)
    )

# Display the final energy summary
lines = min_output.splitlines()
final_index = next(
    (
        index for index, line in enumerate(lines)
        if "FINAL RESULTS" in line
    ),
    None,
)

if final_index is None:
    raise RuntimeError(
        "FINAL RESULTS was not found in the sander output."
    )

print("\nFinal minimisation results\n")
print("\n".join(lines[final_index:final_index + 25]))
print("\nNo numerical or fatal errors were detected.")

# Convert the minimised restart to PDB with ambpdb
with (
    restart_file.open("r") as coordinates,
    minimised_pdb.open("w") as pdb_output,
    ambpdb_log.open("w") as log,
):
    result = subprocess.run(
        ["ambpdb", "-p", str(topology_file)],
        stdin=coordinates,
        stdout=pdb_output,
        stderr=log,
        text=True,
    )

if (
    result.returncode != 0
    or not minimised_pdb.exists()
    or minimised_pdb.stat().st_size == 0
):
    raise RuntimeError(
        f"ambpdb conversion failed. Check {ambpdb_log}"
    )

print(f"\nMinimised PDB: {minimised_pdb}")

# Visualise the minimised structure
view = nv.show_file(str(minimised_pdb))
view.clear_representations()
view.add_cartoon(selection="protein", color="cyan")
view.add_ball_and_stick(selection="[LIG]", color="orange")
view.add_spacefill(selection="_Na or _Cl", radius=0.5)
view.center(selection="protein or [LIG]")
view

Completion checks
  FINAL RESULTS  : True
  Job began      : True
  Run   done     : True
  TIMINGS        : True

Final minimisation results

                    FINAL RESULTS



   NSTEP       ENERGY          RMS            GMAX         NAME    NUMBER
    500      -2.5837E+05     9.8660E-01     9.4534E+01     CD        706

 BOND    =    45073.9416  ANGLE   =      585.3654  DIHED      =     1067.3559
 VDWAALS =    47275.2011  EEL     =  -365451.4695  HBOND      =        0.0000
 1-4 VDW =      888.4059  1-4 EEL =    11986.5176  RESTRAINT  =        0.0000
 CMAP    =      206.4930

--------------------------------------------------------------------------------
   5.  TIMINGS
--------------------------------------------------------------------------------

|    Read coords time           0.03 ( 0.01% of Total)
|                Build the list             3.57 (98.76% of List )
|                Other                      0.04 ( 1.24% of List )
|             List time                  3.61

NGLWidget()